# Train JaxMarl Models in Google Colab
Ensure you are using a GPU runtime (`Runtime` -> `Change runtime type` -> `T4 GPU` or similar).

In [14]:
# Install dependencies
!pip install --upgrade numpy
!pip install jax[cuda12] flax optax wandb
!pip install git+https://github.com/FLAIROx/JaxMARL.git
!pip install navix Pillow matplotlib distrax chex

  Cloning https://github.com/FLAIROx/JaxMARL.git to /tmp/pip-req-build-sc6ashji
  Running command git clone --filter=blob:none --quiet https://github.com/FLAIROx/JaxMARL.git /tmp/pip-req-build-sc6ashji
  Resolved https://github.com/FLAIROx/JaxMARL.git to commit 63226c5fbe5385d67d4b3b97511495a0f2c75bb9
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... canceled
ERROR: Operation cancelled by user


In [15]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import distrax
import chex
import wandb
import navix as nx
import numpy as np
from pathlib import Path
from flax.training.train_state import TrainState
import functools
from typing import Any, Callable, Dict, Tuple, Sequence
from PIL import Image


In [16]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Sequence

class FSQ(nn.Module):
    """
    Finite Scalar Quantization (FSQ) module.
    Projects a continuous vector into a discrete hypercube.
    """
    # A sequence of integers defining the number of levels (L) per dimension (d).
    # e.g., levels=[5, 5, 5] means d=3 dimensions, each with 5 discrete levels.
    levels: Sequence[int]

    @nn.compact
    def __call__(self, z: jnp.ndarray) -> jnp.ndarray:
        """
        Args:
            z: The continuous "thought vector" from the Seer's reasoning module.
               Expected shape: (batch_size, ..., d) where d == len(self.levels)
        Returns:
            z_ste: The quantized vector with gradients preserved via STE.
        """
        levels = jnp.asarray(self.levels, dtype=z.dtype)
        
        # 1. Bounding: Restrict the unbounded input z to the range [-1, 1].
        # We use tanh as a standard, proven method for this projection.
        z_bound = jnp.tanh(z)
        
        # 2. Scaling: Map the [-1, 1] range to the grid [0, L - 1].
        # E.g., for L=5, this maps [-1, 1] to [0, 4].
        half_width = (levels - 1) / 2.0
        z_scaled = z_bound * half_width + half_width
        
        # 3. Quantization: Snap to the nearest integer grid point.
        if self.has_rng('noise'):
            # Inject uniform noise during training to "shake" the FSQ and prevent early mode collapse
            noise = jax.random.uniform(self.make_rng('noise'), z_scaled.shape, minval=-0.2, maxval=0.2)
            z_quantized = jnp.round(z_scaled + noise)
        else:
            z_quantized = jnp.round(z_scaled)
        
        # 4. Straight-Through Estimator (STE) Trick in JAX
        # Forward pass: Evaluates to z_quantized.
        # Backward pass: jax.lax.stop_gradient blocks the gradient from the 
        # non-differentiable z_quantized, so the gradient flows directly through z_scaled.
        z_ste = z_scaled + jax.lax.stop_gradient(z_quantized - z_scaled)
        
        return z_ste

In [17]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Sequence, Tuple

# Import the FSQ module we defined previously

class Seer(nn.Module):
    """
    The 'Hacker' or Prefrontal Cortex network.
    Observes the global state and generates a discrete compositional message.
    """
    fsq_levels: Sequence[int]
    lstm_features: int = 128

    @nn.compact
    def __call__(
        self, 
        carry: Tuple[jnp.ndarray, jnp.ndarray], 
        map_obs: jnp.ndarray, 
        symbolic_obs: jnp.ndarray
    ) -> Tuple[Tuple[jnp.ndarray, jnp.ndarray], jnp.ndarray, jnp.ndarray]:
        """
        Args:
            carry: A tuple of (hidden_state, cell_state) for the LSTM.
            map_obs: The Global Map View grid. Expected shape (batch, H, W, C).
            symbolic_obs: Guard Schedule + Sensor States. Expected shape (batch, features).
            
        Returns:
            new_carry: Updated LSTM state for the next timestep $t+1$.
            discrete_message: The quantized $m_t$ vector sent to the Doer.
            thought_vector: The continuous pre-quantization vector (useful for logging/critic).
        """
        
        # 1. Visual Encoder
        x = nn.Conv(features=32, kernel_size=(3, 3), kernel_init=nn.initializers.orthogonal(jnp.sqrt(2)))(map_obs)
        x = nn.relu(x)
        x = nn.Conv(features=64, kernel_size=(3, 3), kernel_init=nn.initializers.orthogonal(jnp.sqrt(2)))(x)
        x = nn.relu(x)
        
        x_flat = x.reshape((x.shape[0], -1)) 
        
        # 2. Symbolic Encoder
        y = nn.Dense(features=64, kernel_init=nn.initializers.orthogonal(jnp.sqrt(2)))(symbolic_obs)
        y = nn.relu(y)
        
        # 3. Fusion
        fused_features = jnp.concatenate([x_flat, y], axis=-1)
        
        # 4. Reasoning Module: LSTM 
        # Evaluates the current state in the context of the previous timestep [cite: 135]
        lstm_cell = nn.LSTMCell(features=self.lstm_features)
        new_carry, lstm_out = lstm_cell(carry, fused_features)
        
        # 5. Continuous Projection
        # Project LSTM hidden state to continuous vector z of size d.
        # Use Orthogonal init with higher scale to prevent FSQ mode collapse.
        thought_vector = nn.Dense(
            features=len(self.fsq_levels),
            kernel_init=nn.initializers.orthogonal(scale=2.0)
        )(lstm_out)
        
        # 6. Output Head: FSQ Discretizer 
        # Transforms the continuous thought vector into the discrete message $m_t$ 
        fsq = FSQ(levels=self.fsq_levels)
        discrete_message = fsq(thought_vector)
        
        return new_carry, discrete_message, thought_vector

    @staticmethod
    def initialize_carry(batch_size: int, hidden_size: int) -> Tuple[jnp.ndarray, jnp.ndarray]:
        """Utility to generate the initial zero-state for the LSTM at the start of an episode."""
        return (
            jnp.zeros((batch_size, hidden_size)), 
            jnp.zeros((batch_size, hidden_size))
        )

In [18]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Sequence, Tuple

class Doer(nn.Module):
    """
    The 'Thief' or Motor Cortex network.
    Operates on local observations and executes commands via discrete actions.
    """
    fsq_levels: Sequence[int]
    num_actions: int = 6 # e.g., Move N/S/E/W, Toggle, Pick Up
    lstm_features: int = 128
    embed_dim: int = 16

    @nn.compact
    def __call__(
        self,
        carry: Tuple[jnp.ndarray, jnp.ndarray],
        local_obs: jnp.ndarray,
        proprioception: jnp.ndarray,
        message: jnp.ndarray
    ) -> Tuple[Tuple[jnp.ndarray, jnp.ndarray], jnp.ndarray]:
        """
        Args:
            carry: A tuple of (hidden_state, cell_state) for the LSTM.
            local_obs: Egocentric 3x3 grid view. Expected shape (batch, 3, 3, C) or zeros.
            proprioception: Internal states (e.g., carrying item). Expected shape (batch, features).
            message: The quantized $m_t$ vector from the Seer. Expected shape (batch, d).
            
        Returns:
            new_carry: Updated LSTM state for the next timestep $t+1$.
            action_logits: Unnormalized log probabilities for the discrete action space.
        """
        
        # 1. Message Encoder
        message_features = nn.Dense(features=self.embed_dim * len(self.fsq_levels), kernel_init=nn.initializers.orthogonal(jnp.sqrt(2)))(message)
        message_features = nn.relu(message_features)
        
        # 2. Local Visual Encoder
        x = nn.Conv(features=16, kernel_size=(2, 2), kernel_init=nn.initializers.orthogonal(jnp.sqrt(2)))(local_obs)
        x = nn.relu(x)
        x_flat = x.reshape((x.shape[0], -1))
        
        # 3. Proprioception Encoder
        p = nn.Dense(features=16, kernel_init=nn.initializers.orthogonal(jnp.sqrt(2)))(proprioception)
        p = nn.relu(p)
        
        # 4. Fusion
        fused_features = jnp.concatenate([x_flat, p, message_features], axis=-1)
        
        # 5. Reasoning Module
        lstm_cell = nn.LSTMCell(features=self.lstm_features)
        new_carry, lstm_out = lstm_cell(carry, fused_features)
        
        # 6. Output Head: Discrete Action Space
        # Use small scale for action head initial stability
        action_logits = nn.Dense(features=self.num_actions, kernel_init=nn.initializers.orthogonal(0.01))(lstm_out)
        
        return new_carry, action_logits

    @staticmethod
    def initialize_carry(batch_size: int, hidden_size: int) -> Tuple[jnp.ndarray, jnp.ndarray]:
        """Utility to generate the initial zero-state for the LSTM at the start of an episode."""
        return (
            jnp.zeros((batch_size, hidden_size)), 
            jnp.zeros((batch_size, hidden_size))
        )

In [19]:
import jax
import jax.numpy as jnp
import numpy as np
from navix import observations


class NavixGridWrapper:
    """Expose Navix single-agent environments with the Seer/Doer observation split."""

    def __init__(
        self,
        env,
        progress_reward_scale: float = 0.1,
        min_start_distance: float = 0.0,
        step_penalty: float = 0.0,
        bump_penalty: float = 0.1,
        doer_perception_level: int = 0,
    ):
        self._env = env
        self.progress_reward_scale = progress_reward_scale
        self.min_start_distance = jnp.asarray(min_start_distance, dtype=jnp.float32)
        self.step_penalty = jnp.asarray(step_penalty, dtype=jnp.float32)
        self.bump_penalty = jnp.asarray(bump_penalty, dtype=jnp.float32)
        self.doer_perception_level = int(doer_perception_level)

    @property
    def num_actions(self) -> int:
        # Navigation-only action space: turn left, turn right, move forward.
        return 3

    def _split_observations(self, timestep, vision_radius: jnp.ndarray):
        state = timestep.state
        global_map = observations.symbolic(state).astype(jnp.float32)
        full_local_view = observations.symbolic_first_person(state).astype(jnp.float32)

        player = state.get_player()
        goal = state.get_goals()
        center_row = full_local_view.shape[0] // 2
        center_col = full_local_view.shape[1] // 2
        local_view_3x3 = jax.lax.dynamic_slice(
            full_local_view,
            (center_row - 1, center_col - 1, 0),
            (3, 3, full_local_view.shape[-1]),
        )

        symbolic_state = jnp.array(
            [
                player.position[0],
                player.position[1],
                player.direction,
                player.pocket,
                goal.position[0, 0],
                goal.position[0, 1],
            ],
            dtype=jnp.float32,
        )

        proprioception_full = jnp.array(
            [
                player.position[0],
                player.position[1],
                player.direction,
                player.pocket,
            ],
            dtype=jnp.float32,
        )

        if self.doer_perception_level == 0:
            local_view = local_view_3x3
            proprioception = proprioception_full
        elif self.doer_perception_level == 1:
            local_view = jnp.zeros_like(local_view_3x3)
            local_view = local_view.at[0, 1].set(local_view_3x3[0, 1])
            proprioception = proprioception_full
        elif self.doer_perception_level == 2:
            local_view = jnp.zeros_like(local_view_3x3)
            proprioception = jnp.array(
                [
                    player.position[0],
                    player.position[1],
                    0.0,
                    0.0,
                ],
                dtype=jnp.float32,
            )
        elif self.doer_perception_level == 3:
            local_view = jnp.zeros_like(local_view_3x3)
            proprioception = jnp.zeros_like(proprioception_full)
        else:
            raise ValueError(
                f"Unsupported doer_perception_level={self.doer_perception_level}. "
                "Use 0, 1, 2, or 3."
            )

        return {
            "global_map": global_map,
            "symbolic_state": symbolic_state,
            "local_view": local_view,
            "proprioception": proprioception,
        }

    @staticmethod
    def _goal_distance(state) -> jnp.ndarray:
        player = state.get_player()
        goal = state.get_goals()
        return jnp.abs(player.position - goal.position[0]).sum().astype(jnp.float32)

    def reset(self, key: jnp.ndarray, vision_radius: jnp.ndarray = jnp.array(3.0)):
        timestep = self._env.reset(key)
        distance = self._goal_distance(timestep.state)

        def cond_fn(carry):
            _, _, goal_distance = carry
            return goal_distance < self.min_start_distance

        def body_fn(carry):
            rng, _, _ = carry
            rng, sample_key = jax.random.split(rng)
            sampled_timestep = self._env.reset(sample_key)
            goal_distance = self._goal_distance(sampled_timestep.state)
            return rng, sampled_timestep, goal_distance

        _, timestep, _ = jax.lax.while_loop(
            cond_fn,
            body_fn,
            (key, timestep, distance),
        )
        obs = self._split_observations(timestep, vision_radius)
        return obs, timestep

    def reset_batch(self, keys: jnp.ndarray, vision_radius: jnp.ndarray = jnp.array(3.0)):
        return jax.vmap(self.reset, in_axes=(0, None))(keys, vision_radius)

    def step(
        self,
        key: jnp.ndarray,
        timestep,
        action: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(3.0),
    ):
        def reset_branch(_):
            reset_obs, reset_timestep = self.reset(key, vision_radius=vision_radius)
            reward = jnp.asarray(0.0, dtype=jnp.float32)
            done = jnp.asarray(False)
            info = {
                "return": reset_timestep.info.get("return", reward),
                "task_reward": reward,
                "progress_reward": reward,
                "step_penalty": reward,
                "bump_penalty": reward,
                "goal_distance": self._goal_distance(reset_timestep.state),
            }
            return reset_obs, reset_timestep, reward, done, info

        def step_branch(_):
            old_distance = self._goal_distance(timestep.state)
            old_position = timestep.state.get_player().position
            next_timestep = self._env.step(timestep, action)
            new_distance = self._goal_distance(next_timestep.state)
            new_position = next_timestep.state.get_player().position
            obs = self._split_observations(next_timestep, vision_radius)
            task_reward = next_timestep.reward.astype(jnp.float32)
            progress_reward = (old_distance - new_distance) * self.progress_reward_scale
            step_penalty = self.step_penalty
            bumped = jnp.logical_and(
                action == 2,
                jnp.all(new_position == old_position),
            )
            bump_penalty = jnp.where(
                bumped,
                self.bump_penalty,
                jnp.asarray(0.0, dtype=jnp.float32),
            )
            reward = task_reward + progress_reward - step_penalty - bump_penalty
            done = next_timestep.is_done()
            info = dict(next_timestep.info)
            info["task_reward"] = task_reward
            info["progress_reward"] = progress_reward
            info["step_penalty"] = step_penalty
            info["bump_penalty"] = bump_penalty
            info["goal_distance"] = new_distance
            return obs, next_timestep, reward, done, info

        return jax.lax.cond(timestep.is_done(), reset_branch, step_branch, operand=None)

    def step_batch(
        self,
        keys: jnp.ndarray,
        timesteps,
        actions: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(3.0),
    ):
        return jax.vmap(self.step, in_axes=(0, 0, 0, None))(keys, timesteps, actions, vision_radius)

    def render(self, timestep):
        return np.asarray(observations.rgb(timestep.state))


In [20]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax.training.train_state import TrainState
import optax
import chex
from typing import Tuple, Any, Callable
import distrax 
import optax

# 1. Data Structures: Meticulous shape tracking
# Using chex helps enforce accuracy by allowing us to assert shapes later.
@chex.dataclass
class Transition:
    global_obs: chex.Array  # For the Critic (CTDE)
    symbolic_obs: chex.Array # For the Seer
    local_obs: chex.Array   # For the Doer
    proprioception: chex.Array # For the Doer
    message: chex.Array     # For CIC and Heatmap logging
    action: chex.Array
    log_prob: chex.Array
    value: chex.Array
    reward: chex.Array
    task_reward: chex.Array
    progress_reward: chex.Array
    follow_reward: chex.Array
    step_penalty_component: chex.Array
    bump_penalty_component: chex.Array
    done: chex.Array
    advantage: chex.Array
    return_val: chex.Array

# 2. Loss Functions: Separated from network definitions
def calculate_actor_losses(
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    actor_params: dict, # Changed from Any to dict
    transition_batch: Transition,
    init_seer_carry: Tuple[jnp.ndarray, jnp.ndarray], # Changed from Any to Tuple
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray], # Changed from Any to Tuple
    actor_rng: jax.random.PRNGKey,
    clip_eps: float = 0.2,
    entropy_coef: float = 0.01,
    seer_entropy_coef: jnp.ndarray = jnp.array(0.01)
) -> Tuple[jnp.ndarray, dict]:
    """Calculates separate PPO losses for the Seer and Doer reward streams."""
    
    # We must assert shape to prevent silent broadcasting bugs.
    # transition_batch has shape (batch_size, sequence_length, ...) or just (sequence_length, ...)
    # Wait, loop.py returns (num_steps, ...) for single env. Let's assume unbatched or reshape-able.
    # If the user is unrolling over dimension 0:
    
    scan_rngs = jax.random.split(actor_rng, transition_batch.action.shape[0])
    
    def scan_fn(carry, transition_step_and_rng):
        seer_carry, doer_carry = carry
        transition_step, step_rng = transition_step_and_rng
        
        # Seer Forward Pass
        next_seer_carry, discrete_message, thought_vector = seer_apply_fn(
            {"params": actor_params["seer"]},
            seer_carry,
            transition_step.global_obs,
            transition_step.symbolic_obs,
            rngs={"noise": step_rng}
        )
        
        # Doer Forward Pass
        next_doer_carry, logits = doer_apply_fn(
            {"params": actor_params["doer"]},
            doer_carry,
            transition_step.local_obs,
            transition_step.proprioception,
            discrete_message
        )
        return (next_seer_carry, next_doer_carry), (logits, discrete_message, thought_vector)
        
    _, (logits, discrete_messages, thought_vectors) = jax.lax.scan(
        scan_fn, 
        (init_seer_carry, init_doer_carry), 
        (transition_batch, scan_rngs)
    )
    
    pi = distrax.Categorical(logits=logits)
    new_log_probs = pi.log_prob(transition_batch.action)
    entropy = pi.entropy().mean()
    logratio = new_log_probs - transition_batch.log_prob
    ratio = jnp.exp(logratio)

    seer_adv = transition_batch.advantage[..., 0]
    doer_adv = transition_batch.advantage[..., 1]
    seer_adv = (seer_adv - seer_adv.mean()) / (seer_adv.std() + 1e-8)
    doer_adv = (doer_adv - doer_adv.mean()) / (doer_adv.std() + 1e-8)

    seer_loss_unclipped = ratio * seer_adv
    seer_loss_clipped = jnp.clip(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * seer_adv
    seer_actor_loss = -jnp.minimum(seer_loss_unclipped, seer_loss_clipped).mean()

    doer_loss_unclipped = ratio * doer_adv
    doer_loss_clipped = jnp.clip(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * doer_adv
    doer_actor_loss = -jnp.minimum(doer_loss_unclipped, doer_loss_clipped).mean()

    # Calculate aux metrics for information bottleneck auditing
    thought_variance = jnp.var(thought_vectors, axis=0).mean()
    # L2 penalty to prevent pre-tanh values from saturating FSQ
    thought_penalty = jnp.mean(jnp.square(thought_vectors))

    seer_loss = seer_actor_loss + 0.001 * thought_penalty
    doer_loss = doer_actor_loss - entropy_coef * entropy

    return (seer_loss, doer_loss), {
        "seer_actor_loss": seer_actor_loss,
        "doer_actor_loss": doer_actor_loss,
        "entropy": entropy,
        "ratio": ratio.mean(),
        "thought_variance": thought_variance, # Kept for logging visibility
        "discrete_messages": discrete_messages
    }

def calculate_critic_loss(
    critic_apply_fn: Callable,
    critic_params: Any,
    transition_batch: Transition,
    value_clip: float = 0.2
) -> Tuple[jnp.ndarray, dict]:
    """Calculates the value loss for the centralized critic."""
    
    # Assert shape
    chex.assert_rank(transition_batch.global_obs, 4) # (batch_size, H, W, C) since it is flattened over sequence
    
    # The critic uses the global observation (CTDE)
    values = critic_apply_fn({"params": critic_params}, transition_batch.global_obs)

    value_pred_clipped = transition_batch.value + jnp.clip(
        values - transition_batch.value, -value_clip, value_clip
    )
    
    value_losses = jnp.square(values - transition_batch.return_val)
    value_losses_clipped = jnp.square(value_pred_clipped - transition_batch.return_val)
    
    critic_loss = 0.5 * jnp.maximum(value_losses, value_losses_clipped).mean()
    
    return critic_loss, {"critic_loss": critic_loss}

# 3. The Update Step: JIT-compiled gradient application
import functools

@functools.partial(jax.jit, static_argnames=("seer_apply_fn", "doer_apply_fn", "num_ppo_epochs", "num_minibatches"))
def update_actor(
    actor_state: TrainState, 
    transition_batch: Transition,
    init_seer_carry: Any,
    init_doer_carry: Any,
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    seer_entropy_coef: jnp.ndarray,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1
) -> Tuple[TrainState, dict]:
    """Computes gradients and updates the actor network using PPO epochs."""
    
    # Add a batch dimension if missing (assumes trajectory is num_steps, ...)
    # Wait, the prompt says trajectory is (batch_size, seq_len, ...)
    # If the user scales num_envs later, batch_size = num_envs
    
    batch_size = transition_batch.action.shape[0]
    minibatch_size = batch_size // num_minibatches
    
    def epoch_fn(carry, _):
        actor_state, key = carry
        key, subkey = jax.random.split(key)
        
        # Shuffle along the batch dimension
        permutation = jax.random.permutation(subkey, batch_size)
        
        def minibatch_fn(state, start_idx_and_key):
            start_idx, mb_key = start_idx_and_key
            # Slice the minibatch
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], transition_batch)
            
            # Since calculate_actor_loss currently assumes scan over time, and time is dim 1:
            # wait, if input is (batch, time, ...) and scan is over time, we must swap axes!
            # scan_fn expects transition sequence to be the leading dimension.
            # So let's swap seq_len (dim 1) to be dim 0 for scan.
            mb_transition_time_first = jax.tree_util.tree_map(lambda x: jnp.swapaxes(x, 0, 1), mb_transition)
            
            # Extract initial carries for this minibatch
            # Assuming init_seer_carry is (batch_size, ...)
            mb_seer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_seer_carry)
            mb_doer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_doer_carry)
            
            def seer_loss_fn(seer_params):
                (seer_loss, _), metrics = calculate_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": seer_params, "doer": state.params["doer"]},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    mb_key,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return seer_loss, metrics

            def doer_loss_fn(doer_params):
                (_, doer_loss), metrics = calculate_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": state.params["seer"], "doer": doer_params},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    mb_key,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return doer_loss, metrics

            (seer_loss, seer_metrics), seer_grads = jax.value_and_grad(
                seer_loss_fn,
                has_aux=True,
            )(state.params["seer"])
            (doer_loss, doer_metrics), doer_grads = jax.value_and_grad(
                doer_loss_fn,
                has_aux=True,
            )(state.params["doer"])

            grads = {"seer": seer_grads, "doer": doer_grads}

            # Record explicit gradient norms for auditing
            seer_grad_norm = optax.global_norm(grads["seer"])
            doer_grad_norm = optax.global_norm(grads["doer"])

            metrics = {
                "seer_loss": seer_loss,
                "doer_loss": doer_loss,
                "seer_actor_loss": seer_metrics["seer_actor_loss"],
                "doer_actor_loss": doer_metrics["doer_actor_loss"],
                "entropy": doer_metrics["entropy"],
                "ratio": doer_metrics["ratio"],
                "thought_variance": seer_metrics["thought_variance"],
                "discrete_messages": seer_metrics["discrete_messages"],
                "seer_grad_norm": seer_grad_norm,
                "doer_grad_norm": doer_grad_norm,
            }
            
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics
            
        # Minibatch loop (scan over start_indices)
        start_indices = jnp.arange(0, batch_size, minibatch_size)
        mb_keys = jax.random.split(subkey, start_indices.shape[0])
        actor_state, mb_metrics = jax.lax.scan(minibatch_fn, actor_state, (start_indices, mb_keys))
        
        # Average metrics over minibatches
        epoch_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in mb_metrics.items()}
        return (actor_state, key), epoch_metrics
        
    (final_actor_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn, (actor_state, rng), None, length=num_ppo_epochs
    )
    
    # Return averaged metrics over epochs
    final_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in epoch_metrics.items()}
    return final_actor_state, final_metrics

@functools.partial(jax.jit, static_argnames=("critic_apply_fn", "num_ppo_epochs", "num_minibatches"))
def update_critic(
    critic_state: TrainState, 
    transition_batch: Transition,
    critic_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1
) -> Tuple[TrainState, dict]:
    """Computes gradients and updates the critic network."""
    
    # For critic we can flatten the batch and sequence dimension since it's just MLPs
    batch_size = transition_batch.action.shape[0] * transition_batch.action.shape[1]
    flat_transition = jax.tree_util.tree_map(lambda x: x.reshape((batch_size,) + x.shape[2:]), transition_batch)
    minibatch_size = batch_size // num_minibatches
    
    def epoch_fn(carry, _):
        critic_state, key = carry
        key, subkey = jax.random.split(key)
        permutation = jax.random.permutation(subkey, batch_size)
        
        def minibatch_fn(state, start_idx):
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], flat_transition)
            
            loss_fn = lambda params: calculate_critic_loss(
                critic_apply_fn, params, mb_transition
            )
            
            (loss, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics
            
        start_indices = jnp.arange(0, batch_size, minibatch_size)
        critic_state, mb_metrics = jax.lax.scan(minibatch_fn, critic_state, start_indices)
        
        epoch_metrics = jax.tree_util.tree_map(lambda x: x.mean(), mb_metrics)
        return (critic_state, key), epoch_metrics

    (final_critic_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn, (critic_state, rng), None, length=num_ppo_epochs
    )
    
    final_metrics = jax.tree_util.tree_map(lambda x: x.mean(), epoch_metrics)
    return final_critic_state, final_metrics


In [21]:
import jax
import jax.numpy as jnp
from typing import Tuple

def compute_gae(
    rewards: jnp.ndarray,
    values: jnp.ndarray,
    dones: jnp.ndarray,
    last_val: jnp.ndarray,
    gamma: float = 0.99,
    gae_lambda: float = 0.95
) -> Tuple[jnp.ndarray, jnp.ndarray]:
    """
    Computes Generalized Advantage Estimation (GAE) and target returns.
    
    Args:
        rewards: Array of rewards collected during the rollout. Shape: (num_steps,)
        values: Array of value predictions from the Critic. Shape: (num_steps,)
        dones: Array of boolean/integer done flags. Shape: (num_steps,)
        last_val: The Critic's value prediction for the state *after* the final step.
        gamma: Discount factor.
        gae_lambda: Bias-variance tradeoff parameter for GAE.
        
    Returns:
        advantages: The calculated GAE advantages. Shape: (num_steps,)
        returns: The target values for the Critic to learn from (Advantages + Values).
    """
    
    def _gae_step(carry, transition_data):
        """A single step of the reverse scan."""
        gae_t_plus_1 = carry
        reward, value, done, next_value = transition_data
        
        # Calculate the Temporal Difference (TD) error
        # If the episode ended (done=1), the next state has no value.
        delta = reward + gamma * next_value * (1.0 - done) - value
        
        # Calculate the advantage for the current timestep
        gae_t = delta + gamma * gae_lambda * (1.0 - done) * gae_t_plus_1
        
        # Pass the current advantage back as the carry for the previous timestep
        return gae_t, gae_t

    # To calculate the TD error, we need the value of the next state for every step.
    # We create an array of "next values" by shifting the values array by one 
    # and appending the bootstrap 'last_val' at the end.
    next_values = jnp.append(values[1:], last_val)
    
    # Pack the data for the scan
    scan_data = (rewards, values, dones, next_values)
    
    # Initialize the carry with 0.0 (the advantage after the final step)
    initial_gae = 0.0
    
    # Run the scan in reverse to propagate advantages backwards
    _, advantages = jax.lax.scan(_gae_step, initial_gae, scan_data, reverse=True)
    
    # The return value (target for the critic) is simply the advantage + the predicted value
    returns = advantages + values
    
    return advantages, returns

In [22]:
import jax
import jax.numpy as jnp
import distrax
from typing import Tuple, Any, Dict


def make_rollout_step(
    env,
    seer_apply_fn,
    doer_apply_fn,
    critic_apply_fn,
    follow_reward_scale=0.1,
):
    """
    A closure that returns the JAX-compilable step function.
    Passing the environment and apply functions here avoids passing them 
    repeatedly into the compiled loop.
    """
    follow_reward_scale = jnp.asarray(follow_reward_scale, dtype=jnp.float32)
    
    def rollout_step(runner_state: Tuple, _):
        """
        Executes a single environment step and network forward pass.
        Designed to be passed directly to jax.lax.scan.
        """
        # Unpack the runner state
        params, seer_carry, doer_carry, env_state, env_obs, rng, vision_radius = runner_state
        num_envs = env_obs["global_map"].shape[0]
        
        # Split the PRNG key for the stochastic actions
        rng, doer_rng, env_rng = jax.random.split(rng, 3)
        doer_rng, seer_rng = jax.random.split(doer_rng)
        env_step_keys = jax.random.split(env_rng, num_envs)
        
        # 1. Seer Forward Pass (Prefrontal Cortex)
        # Enforcing CTDE: Seer gets the global view[cite: 131, 132].
        # In a custom jaxmarl wrapper, you would extract these from env_obs
        global_map = env_obs["global_map"]
        symbolic_obs = env_obs["symbolic_state"]
        
        next_seer_carry, discrete_message, _ = seer_apply_fn(
            {"params": params["seer"]}, 
            seer_carry, 
            global_map, 
            symbolic_obs,
            rngs={"noise": seer_rng}
        )
        
        # 2. Doer Forward Pass (Motor Cortex)
        # Enforcing Functional Asymmetry: Doer gets local view and the message[cite: 137, 138].
        local_obs = env_obs["local_view"]
        proprioception = env_obs["proprioception"]
        
        next_doer_carry, action_logits = doer_apply_fn(
            {"params": params["doer"]}, 
            doer_carry, 
            local_obs, 
            proprioception, 
            discrete_message
        )
        _, null_action_logits = doer_apply_fn(
            {"params": params["doer"]},
            doer_carry,
            local_obs,
            proprioception,
            jnp.zeros_like(discrete_message),
        )
        
        # 3. Action Selection
        pi = distrax.Categorical(logits=action_logits)
        null_pi = distrax.Categorical(logits=null_action_logits)
        action = pi.sample(seed=doer_rng)
        log_prob = pi.log_prob(action)
        follow_reward = jnp.maximum(
            pi.log_prob(action) - null_pi.log_prob(action),
            0.0,
        ) * follow_reward_scale
        
        # 4. Critic Forward Pass (Centralized Training)
        # The critic evaluates the global state to guide learning[cite: 111].
        value = critic_apply_fn({"params": params["critic"]}, global_map)
        
        # 5. Environment Step
        # Step the jaxmarl environment using the chosen action
        # Note: jaxmarl expects a dictionary of actions for multi-agent, 
        # adapt this based on your specific wrapper implementation.
        next_env_obs, next_env_state, reward, done, info = env.step_batch(
            env_step_keys, env_state, action, vision_radius=vision_radius
        )

        task_reward = info["task_reward"]
        progress_reward = info["progress_reward"]
        step_penalty = info["step_penalty"]
        bump_penalty = info["bump_penalty"]
        
        # Conditional Follow Reward: Only reward listening to Seer if it actually helped
        actual_follow_reward = jnp.where(progress_reward > 0, follow_reward, 0.0)
        
        seer_reward = task_reward + progress_reward - step_penalty
        doer_reward = task_reward + progress_reward + actual_follow_reward - step_penalty - bump_penalty
        reward = jnp.stack([seer_reward, doer_reward], axis=-1)

        done_mask = done[:, None]
        next_seer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done_mask, jnp.zeros_like(x), x),
            next_seer_carry,
        )
        next_doer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done_mask, jnp.zeros_like(x), x),
            next_doer_carry,
        )
        
        # 6. Build the Transition
        transition = Transition(
            global_obs=global_map,
            symbolic_obs=symbolic_obs,
            local_obs=local_obs,
            proprioception=proprioception,
            message=discrete_message,
            action=action,
            log_prob=log_prob,
            value=value,
            reward=reward,
            task_reward=task_reward,
            progress_reward=progress_reward,
            follow_reward=follow_reward,
            step_penalty_component=step_penalty,
            bump_penalty_component=bump_penalty,
            done=done,
            # Advantage and return will be calculated post-rollout using GAE
            advantage=jnp.zeros_like(reward), 
            return_val=jnp.zeros_like(reward)
        )
        
        # Repack the updated runner state
        next_runner_state = (
            params, next_seer_carry, next_doer_carry, next_env_state, next_env_obs, rng, vision_radius
        )
        
        return next_runner_state, transition

    return rollout_step

import functools

@functools.partial(jax.jit, static_argnames=("num_steps", "step_fn", "critic_apply_fn", "doer_apply_fn"))
def generate_trajectory_and_gae(
    params, rng, env_obs, env_state, seer_carry, doer_carry, vision_radius: jnp.ndarray, cic_coef: jnp.ndarray, num_steps: int,
    step_fn, critic_apply_fn, doer_apply_fn
):
    """
    Executes the full episode rollout and computes GAE in a single compiled pass.
    Note: We pass the pre-compiled step_fn and initial states directly here 
    for better JAX compilation efficiency.
    """
    initial_runner_state = (params, seer_carry, doer_carry, env_state, env_obs, rng, vision_radius)
    
    # 1. Execute the scan loop to collect the raw trajectory
    final_runner_state, trajectory_batch = jax.lax.scan(
        step_fn, initial_runner_state, None, length=num_steps
    )
    
    # 2. Extract the final state for Critic bootstrapping
    # Unpack the final runner state to get the last env_obs
    _, _, _, _, final_env_obs, _, _ = final_runner_state
    
    # Enforce CTDE: The critic evaluates the global map 
    final_global_map = final_env_obs["global_map"]
    
    # 3. Calculate the bootstrap value (last_val)
    # The critic evaluates the state *after* the final step
    last_val = critic_apply_fn({"params": params["critic"]}, final_global_map)
    
    reward_with_cic = trajectory_batch.reward
    
    # 5. Compute GAE
    # Note: If you scale up to multiple environments (num_envs > 1), you would wrap 
    # compute_gae with jax.vmap(compute_gae, in_axes=1, out_axes=1) to vectorize 
    # the advantage calculation across all environments simultaneously.
    advantages, returns = jax.vmap(
        jax.vmap(
            compute_gae,
            in_axes=(1, 1, 1, 0, None, None),
            out_axes=1,
        ),
        in_axes=(2, 2, None, 1, None, None),
        out_axes=2,
    )(
        reward_with_cic,
        trajectory_batch.value,
        trajectory_batch.done,
        last_val,
        0.99,
        0.95,
    )
    
    # 5. Update the trajectory batch
    # Using the .replace() method provided by chex/flax dataclasses
    trajectory_batch = trajectory_batch.replace(
        advantage=advantages,
        return_val=returns
    )
    
    return final_runner_state, trajectory_batch


In [23]:
import jax
import jax.numpy as jnp
from typing import Tuple, Callable
import functools

@functools.partial(jax.jit, static_argnames=("doer_apply_fn",))
def compute_cic(
    doer_apply_fn: Callable,
    doer_params: dict,
    transition_batch, 
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray],
    rng: jax.random.PRNGKey
) -> jnp.ndarray:
    """
    Computes Causal Influence of Communication (CIC) by ablating the Seer's message
    and measuring the divergence in the Doer's resulting deterministic actions.
    """
    
    def scan_fn(carry, step_data):
        doer_carry = carry
        msg, loc, prop = step_data
        
        next_doer_carry, logits = doer_apply_fn(
            {"params": doer_params},
            doer_carry,
            loc,
            prop,
            msg
        )
        return next_doer_carry, logits

    # Transition batch shapes: (seq_len, num_envs, ...) from loop.py
    loc = transition_batch.local_obs
    prop = transition_batch.proprioception
    msg_true = transition_batch.message

    # True forward pass
    _, true_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_true, loc, prop))
    
    # Ablated forward pass: shuffle messages independently over time for each env.
    env_keys = jax.random.split(rng, msg_true.shape[1])
    msg_shuffled = jax.vmap(
        lambda key, msgs: jax.random.permutation(key, msgs, axis=0)
    )(env_keys, jnp.swapaxes(msg_true, 0, 1))
    msg_shuffled = jnp.swapaxes(msg_shuffled, 0, 1)
    
    _, ablated_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_shuffled, loc, prop))
    
    # Calculate CIC: divergence in deterministic policy
    true_actions = jnp.argmax(true_logits, axis=-1)
    ablated_actions = jnp.argmax(ablated_logits, axis=-1)
    
    cic = jnp.mean((true_actions != ablated_actions).astype(jnp.float32))
    
    return cic


In [24]:
from pathlib import Path

import jax
import jax.numpy as jnp
from PIL import Image


def visualize_episode(
    env,
    params,
    rng,
    seer,
    doer,
    filename="episode.gif",
    vision_radius=jnp.array(2.0),
    max_steps=200,
):
    """Run one greedy evaluation episode and save it as a GIF."""
    frames = []
    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    rng, reset_rng = jax.random.split(rng)
    obs, state = env.reset(reset_rng, vision_radius=vision_radius)

    seer_carry = seer.initialize_carry(batch_size=1, hidden_size=128)
    doer_carry = doer.initialize_carry(batch_size=1, hidden_size=128)

    done = False
    solved = False
    step_count = 0

    while not bool(done) and step_count < max_steps:
        frame = env.render(state)
        frames.append(Image.fromarray(frame))

        global_map = obs["global_map"][None, ...]
        symbolic_state = obs["symbolic_state"][None, ...]
        local_view = obs["local_view"][None, ...]
        proprioception = obs["proprioception"][None, ...]

        seer_carry, message, _ = seer.apply(
            {"params": params["seer"]},
            seer_carry,
            global_map,
            symbolic_state,
        )

        doer_carry, action_logits = doer.apply(
            {"params": params["doer"]},
            doer_carry,
            local_view,
            proprioception,
            message,
        )

        action = jnp.argmax(action_logits[0]).astype(jnp.int32)

        rng, step_rng = jax.random.split(rng)
        obs, state, reward, done, info = env.step(
            step_rng,
            state,
            action,
            vision_radius=vision_radius,
        )
        solved = solved or bool(done) and float(info["task_reward"]) > 0.0

        step_count += 1

    if not frames:
        raise RuntimeError("Visualization episode produced no frames.")

    frames[0].save(
        output_path,
        save_all=True,
        append_images=frames[1:],
        duration=120,
        loop=0,
    )
    print(f"Episode saved to {output_path}")
    return output_path, solved


In [25]:
@chex.dataclass
class RunnerState:
    actor_state: TrainState
    critic_state: TrainState
    seer_carry: jnp.ndarray
    doer_carry: jnp.ndarray
    env_state: Any
    env_obs: Any
    rng: jax.random.PRNGKey
    update: int
    success_streak: int

# For the sake of a complete script, here is a pragmatic, standard Critic
class GlobalCritic(nn.Module):
    """Evaluates the global state to guide learning (CTDE)."""
    @nn.compact
    def __call__(self, global_map: jnp.ndarray) -> jnp.ndarray:
        x = nn.Conv(features=32, kernel_size=(3, 3), kernel_init=nn.initializers.orthogonal(jnp.sqrt(2)))(global_map)
        x = nn.relu(x)
        x = nn.Conv(features=64, kernel_size=(3, 3), kernel_init=nn.initializers.orthogonal(jnp.sqrt(2)))(x)
        x = nn.relu(x)
        x = x.reshape((x.shape[0], -1))
        x = nn.Dense(features=128, kernel_init=nn.initializers.orthogonal(jnp.sqrt(2)))(x)
        x = nn.relu(x)
        # Small scale for value head to stabilize early training
        value = nn.Dense(features=2, kernel_init=nn.initializers.orthogonal(0.01))(x)
        return value

def main():
    # 1. Configuration and Logging
    config = {
        "learning_rate": 3e-4,
        "num_envs": 256,  # Increased for massive batch size -> critic stability
        "num_steps": 128,
        "total_timesteps": 10_000_000, # Increased scale
        "env_id": "Navix-Empty-8x8-v0",
        "fsq_levels": [4],
        "seed": 42,
        "follow_reward_scale": 0.1,
        "progress_reward_scale": 0.2,
        "cic_coef": 0.0,
        "doer_perception_level": 3,
        "curriculum_success_streak": 3,
        "min_start_distance": 1.0,
        "step_penalty": 0.01,
        "bump_penalty": 0.1,
        "visualize_every": 20,
        "visualize_max_steps": 10,
        "visualize_dir": "artifacts/episodes",
        "log_every": 1,
        "log_distribution_every": 5,
        "log_heatmap_every": 20,
        "log_cic_every": 10,
    }
    
    wandb.init(entity="eleftheriaklk-ucl", project="brian_test_optimized", config=config)
    
    # 2. PRNG Key Initialization
    rng = jax.random.PRNGKey(config["seed"])
    rng, seer_init_rng, doer_init_rng, critic_init_rng, env_rng = jax.random.split(rng, 5)

    # 3. Environment Instantiation
    raw_env = nx.make(config["env_id"])
    env = NavixGridWrapper(
        raw_env,
        progress_reward_scale=config["progress_reward_scale"],
        min_start_distance=config["min_start_distance"],
        step_penalty=config["step_penalty"],
        bump_penalty=config["bump_penalty"],
        doer_perception_level=config["doer_perception_level"],
    )

    # 4. Initial Environment Reset
    reset_rng, env_rng = jax.random.split(env_rng, 2)
    reset_keys = jax.random.split(reset_rng, config["num_envs"])
    env_obs, env_state = env.reset_batch(reset_keys, vision_radius=jnp.array(3.0))

    # 5. Network Instantiation
    seer = Seer(fsq_levels=config["fsq_levels"])
    doer = Doer(fsq_levels=config["fsq_levels"], num_actions=env.num_actions)
    critic = GlobalCritic()

    # 6. Parameter Initialization
    env_map = env_obs["global_map"][:1]
    env_sym = env_obs["symbolic_state"][:1]
    env_local = env_obs["local_view"][:1]
    env_prop = env_obs["proprioception"][:1]
    env_msg = jnp.zeros((1, len(config["fsq_levels"])))
    
    init_seer_carry_single = seer.initialize_carry(1, 128)
    init_doer_carry_single = doer.initialize_carry(1, 128)

    seer_params = seer.init(seer_init_rng, init_seer_carry_single, env_map, env_sym)["params"]
    doer_params = doer.init(doer_init_rng, init_doer_carry_single, env_local, env_prop, env_msg)["params"]
    critic_params = critic.init(critic_init_rng, env_map)["params"]

    seer_carry = seer.initialize_carry(config["num_envs"], 128)
    doer_carry = doer.initialize_carry(config["num_envs"], 128)

    # 7. Optimizer and TrainState Setup
    tx = optax.chain(
        optax.clip_by_global_norm(0.5),
        optax.adam(learning_rate=config["learning_rate"], eps=1e-5)
    )

    actor_state = TrainState.create(
        apply_fn=None,
        params={"seer": seer_params, "doer": doer_params},
        tx=tx
    )
    
    critic_state = TrainState.create(
        apply_fn=critic.apply,
        params=critic_params,
        tx=tx
    )

    step_fn = make_rollout_step(
        env,
        seer.apply,
        doer.apply,
        critic.apply,
        follow_reward_scale=config["follow_reward_scale"],
    )

    # 8. Define the training step for jax.lax.scan
    def train_step(runner_state: RunnerState, _):
        # A. Curriculums
        update = runner_state.update
        # Restore constants for stability as requested by the user previously
        vision_radius = 3.0 
        seer_entropy_coef = 0.1 
        
        # B. Collect Trajectory
        rng, rollout_rng = jax.random.split(runner_state.rng)
        
        params = {
            "seer": runner_state.actor_state.params["seer"],
            "doer": runner_state.actor_state.params["doer"],
            "critic": runner_state.critic_state.params
        }
        
        final_rollout_state, trajectory_batch = generate_trajectory_and_gae(
            params,
            rollout_rng,
            runner_state.env_obs,
            runner_state.env_state,
            runner_state.seer_carry,
            runner_state.doer_carry,
            vision_radius,
            jnp.array(config["cic_coef"], dtype=jnp.float32),
            config["num_steps"],
            step_fn, critic.apply, doer.apply
        )
        
        # C. Update Networks
        rng, actor_rng, critic_rng = jax.random.split(rng, 3)
        
        batched_trajectory = jax.tree_util.tree_map(
            lambda x: jnp.swapaxes(x, 0, 1),
            trajectory_batch
        )
        
        actor_state, actor_metrics = update_actor(
            runner_state.actor_state, 
            batched_trajectory, 
            runner_state.seer_carry, 
            runner_state.doer_carry, 
            seer.apply, doer.apply, actor_rng, seer_entropy_coef
        )
        critic_state, critic_metrics = update_critic(
            runner_state.critic_state, 
            batched_trajectory, 
            critic.apply, critic_rng
        )
        
        # D. Update Runner State
        _, next_seer_carry, next_doer_carry, next_env_state, next_env_obs, _, _ = final_rollout_state
        
        next_state = RunnerState(
            actor_state=actor_state,
            critic_state=critic_state,
            seer_carry=next_seer_carry,
            doer_carry=next_doer_carry,
            env_state=next_env_state,
            env_obs=next_env_obs,
            rng=rng,
            update=update + 1,
            success_streak=runner_state.success_streak
        )
        
        metrics = {
            "update": update,
            "seer_loss": actor_metrics.get("seer_loss", 0.0),
            "doer_loss": actor_metrics.get("doer_loss", 0.0),
            "entropy": actor_metrics.get("entropy", 0.0),
            "critic_loss": critic_metrics.get("critic_loss", 0.0),
            "seer_reward": trajectory_batch.reward[..., 0].mean(),
            "doer_reward": trajectory_batch.reward[..., 1].mean(),
            "task_reward": trajectory_batch.task_reward.mean(),
            "progress_reward": trajectory_batch.progress_reward.mean(),
            "follow_reward": trajectory_batch.follow_reward.mean(),
            "vision_radius": vision_radius,
            "seer_entropy_coef": seer_entropy_coef,
            "thought_variance": actor_metrics.get("thought_variance", 0.0),
            "discrete_messages": actor_metrics.get("discrete_messages"),
            "trajectory_batch": trajectory_batch, # Partially return for CIC calculation outside scan if needed
        }
        
        return next_state, metrics

    # 9. Initial Runner State
    runner_state = RunnerState(
        actor_state=actor_state,
        critic_state=critic_state,
        seer_carry=seer_carry,
        doer_carry=doer_carry,
        env_state=env_state,
        env_obs=env_obs,
        rng=rng,
        update=0,
        success_streak=0
    )

    # Compile the training block
    train_block = jax.jit(lambda state: jax.lax.scan(train_step, state, None, length=config["log_every"]))

    # 10. The Main Training Loop (Python side handles logging and viz)
    total_updates = config["total_timesteps"] // (config["num_steps"] * config["num_envs"])
    num_blocks = total_updates // config["log_every"]
    
    print(f"Starting optimized training with {config['num_envs']} envs...")
    
    for block in range(num_blocks):
        runner_state, metrics = train_block(runner_state)
        
        # Extract the last update's metrics for logging
        last_metrics = jax.tree_util.tree_map(lambda x: x[-1], metrics)
        update_idx = int(last_metrics["update"])
        
        # A. Logging to WandB
        log_dict = {
            "update": update_idx,
            "seer_loss": last_metrics["seer_loss"],
            "doer_loss": last_metrics["doer_loss"],
            "entropy": last_metrics["entropy"],
            "critic_loss": last_metrics["critic_loss"],
            "seer_reward": last_metrics["seer_reward"],
            "doer_reward": last_metrics["doer_reward"],
            "task_reward": last_metrics["task_reward"],
            "progress_reward": last_metrics["progress_reward"],
            "follow_reward": last_metrics["follow_reward"],
            "seer_entropy_coef": last_metrics["seer_entropy_coef"],
            "curriculum_level": env.doer_perception_level,
        }

        # B. Periodic Evaluation and Visualization
        if update_idx > 0 and update_idx % config["log_distribution_every"] == 0:
            print(f"Update {update_idx} | Seer/Doer Loss: {last_metrics['seer_loss']:.4f}/{last_metrics['doer_loss']:.4f} | Task Reward: {last_metrics['task_reward']:.4f}")
            
            # Log message distribution as a histogram
            msgs = metrics["discrete_messages"].reshape(-1) # Flatten all messages in the block
            log_dict["message_distribution"] = wandb.Histogram(np.array(msgs))

        # C. Signal-to-Action Heatmap (Auditing if discrete symbols correlate with actions)
        if update_idx > 0 and update_idx % config["log_heatmap_every"] == 0:
            traj = metrics["trajectory_batch"]
            m = np.array(traj.message).reshape((-1, len(config["fsq_levels"])))
            a = np.array(traj.action).reshape(-1)
            
            levels = np.array(config["fsq_levels"])
            if len(levels) == 1:
                m_idx = m[:, 0]
            else:
                multipliers = np.cumprod(np.concatenate([np.array([1]), levels[:-1]]))
                m_idx = (m * multipliers).sum(axis=-1)
            
            num_message_codes = int(np.prod(levels))
            num_actions = env.num_actions
            H, _, _ = np.histogram2d(
                m_idx, a, 
                bins=[np.arange(num_message_codes + 1), np.arange(num_actions + 1)]
            )
            log_dict["signal_action_heatmap"] = wandb.Image(H / (H.max() + 1e-8))

        if update_idx > 0 and update_idx % config["visualize_every"] == 0:
            params = {
                "seer": runner_state.actor_state.params["seer"],
                "doer": runner_state.actor_state.params["doer"],
                "critic": runner_state.critic_state.params
            }
            rng, viz_rng = jax.random.split(runner_state.rng)
            viz_path = Path(config["visualize_dir"]) / f"episode_{update_idx:05d}.gif"
            _, solved = visualize_episode(
                env, params, viz_rng, seer, doer,
                filename=str(viz_path),
                vision_radius=3.0,
                max_steps=config["visualize_max_steps"],
            )
            
            # Log Video to WandB
            log_dict.update({
                "episode_viz": wandb.Video(str(viz_path), fps=4, format="gif"),
                "episode_solved": float(solved)
            })

        if update_idx > 0 and update_idx % config["log_cic_every"] == 0:
            # Compute and log CIC score (Seer Score)
            rng, cic_rng = jax.random.split(runner_state.rng)
            last_traj = jax.tree_util.tree_map(lambda x: x[-1], metrics["trajectory_batch"])
            cic_score = compute_cic(
                doer.apply, runner_state.actor_state.params["doer"], last_traj, runner_state.doer_carry, cic_rng
            )
            log_dict["CIC_Score"] = cic_score

        # D. Curriculum Check (Decoupled from visualization)
        # Average number of maze completions per environment during this block
        traj_batch = metrics["trajectory_batch"]
        completions_per_env = np.sum(traj_batch.task_reward) / config["num_envs"]
        
        # If at least 80% of envs completed the maze at least once
        if completions_per_env >= 0.8:
            runner_state = runner_state.replace(success_streak=runner_state.success_streak + 1)
        else:
            runner_state = runner_state.replace(success_streak=0)
            
        log_dict["success_streak"] = runner_state.success_streak
        log_dict["completions_per_env"] = completions_per_env

        # Finally, upload everything!
        wandb.log(log_dict)

        # Manual Level advancement logic
        if runner_state.success_streak >= config["curriculum_success_streak"] and env.doer_perception_level < 3:
            env.doer_perception_level += 1
            curr_level = env.doer_perception_level
            print(f"Advancing to perception level {curr_level}")
            
            # Re-compile step_fn with new env setting (or make level a dyn param if preferred)
            step_fn = make_rollout_step(env, seer.apply, doer.apply, critic.apply, follow_reward_scale=config["follow_reward_scale"])
            train_block = jax.jit(lambda state: jax.lax.scan(train_step, state, None, length=config["log_every"]))
            
            # Reset envs for the new level
            rng, env_rng = jax.random.split(runner_state.rng, 2)
            reset_keys = jax.random.split(env_rng, config["num_envs"])
            new_obs, new_state = env.reset_batch(reset_keys, vision_radius=jnp.array(3.0))
            runner_state = runner_state.replace(env_obs=new_obs, env_state=new_state, rng=rng)

if __name__ == "__main__":
    main()


CIC_Score,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
completions_per_env,▁▂▃▃▃▁▄▃▄▅▅▄▃▅▃▇▅▆▅▄▄▅▄▆▆▆▅▄▄▄▅▅▄█▆▆▄▂▃▆
critic_loss,▁▃▂▂▃▃▄▅▅▅▃▃▃▄▄▄▄▆▅▅▅▅▅▆▅▄▅▆▆▆▅▅▇█▆▆▃▄▃▅
curriculum_level,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
doer_loss,▅▅▅▃▃▁█▃▂▅▁▄▃▄▄▅▄▄▃▅▆▂▃▄▄▄▅▅▃▃▄▅▄▄▃▄▆▃▅▃
doer_reward,▄▂▄▂▇▄▃▄▅▃▂▇▄▄▅▆▃▆▃▄▆▂▅▄▅▄█▃▇▇▅▅▅▅▃▃▁▅▄▆
entropy,██▇▇▆▄▂▄▃▄▄▃▄▃▅▇▇▄▃▃▅▅▅▅▃▃▃▃▄▄▃▃▃▃▃▅▆▁▁▄
episode_solved,▁▁▁▁▁▁▁▁▁▁▁
follow_reward,▁▃▄▅▆▇▇▇▇▇▇▇▆▆▆▇▇▇▇▇▇▇▇▇█████████▇▇█████
progress_reward,▂▁▂▄▃▃▄▅▄█▅▆▅▃▄▃▄▄▄▆▆▄▅▄▅▇▆▅▆█▄▄▄▄▄▂▃▇▆▆
+6,...


Starting optimized training with 256 envs...
Update 5 | Seer/Doer Loss: -0.0000/-0.0111 | Task Reward: 0.0020
Update 10 | Seer/Doer Loss: 0.0003/-0.0111 | Task Reward: 0.0021
Update 15 | Seer/Doer Loss: 0.0005/-0.0111 | Task Reward: 0.0026
Update 20 | Seer/Doer Loss: 0.0004/-0.0110 | Task Reward: 0.0025


wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


Episode saved to artifacts/episodes/episode_00020.gif
Update 25 | Seer/Doer Loss: 0.0001/-0.0111 | Task Reward: 0.0031
Update 30 | Seer/Doer Loss: 0.0001/-0.0110 | Task Reward: 0.0034
Update 35 | Seer/Doer Loss: 0.0004/-0.0110 | Task Reward: 0.0031
Update 40 | Seer/Doer Loss: 0.0003/-0.0108 | Task Reward: 0.0030


wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


...

Episode saved to artifacts/episodes/episode_00260.gif
Update 265 | Seer/Doer Loss: 0.0158/0.0387 | Task Reward: 0.0707
Update 270 | Seer/Doer Loss: 0.0038/0.0304 | Task Reward: 0.0598
Update 275 | Seer/Doer Loss: 0.0116/0.0253 | Task Reward: 0.0686
Update 280 | Seer/Doer Loss: 0.0098/0.0120 | Task Reward: 0.0744


wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


Episode saved to artifacts/episodes/episode_00280.gif
Update 285 | Seer/Doer Loss: 0.0023/0.0034 | Task Reward: 0.0781
Update 290 | Seer/Doer Loss: 0.0178/0.0014 | Task Reward: 0.0776
Update 295 | Seer/Doer Loss: 0.0033/0.0018 | Task Reward: 0.0702
Update 300 | Seer/Doer Loss: 0.0077/0.0130 | Task Reward: 0.0753


wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


Episode saved to artifacts/episodes/episode_00300.gif


In [ ]:
wandb.login()

In [ ]:
main()